# `00_download_eval_datasets.ipynb` — one-time download of the 7 eval datasets

Run this ONCE per HuggingFace account (Colab or Kaggle). It downloads all 7 hallucination-detection
evaluation datasets and saves them as parquet files. Then upload those parquet files alongside each
model's `dataset_full.json` when running `02_all_variants.ipynb` and `03_baselines_sota.ipynb`.

**Why this exists**

The 7 dataset loaders are repeated in every `02` and `03` notebook (and we have 6 large models × 2 analysis notebooks = 12 places where it would run). HuggingFace rate-limits unauthenticated downloads, occasionally renames namespaces, and the dataset configs can take ~5 min to resolve fresh. Doing this once and uploading the parquet files makes subsequent runs ~30× faster and immune to HF reachability issues.

**Output files (current working dir):**

| File | Source |
|---|---|
| `eval_truthfulqa.parquet` | `truthfulqa/truthful_qa` generation split |
| `eval_triviaqa.parquet` | `mandarjoshi/trivia_qa` rc.nocontext validation |
| `eval_coqa.parquet` | `stanfordnlp/coqa` validation |
| `eval_tydiqa.parquet` | `google-research-datasets/tydiqa` secondary_task English |
| `eval_halueval_qa.parquet` | `pminervini/HaluEval` qa |
| `eval_halueval_summ.parquet` | `pminervini/HaluEval` summarization |
| `eval_halueval_dialog.parquet` | `pminervini/HaluEval` dialogue |
| `eval_nq_open.parquet` | `google-research-datasets/nq_open` validation (~3.6k samples) |
| `eval_hotpotqa.parquet` | `hotpotqa/hotpot_qa` distractor validation (~7.4k samples) |
| `eval_popqa.parquet` | `akariasai/PopQA` test (~14k samples) |
| `eval_datasets.zip` | All 10 above, bundled |

**How to use after running:**

* **Kaggle**: take `eval_datasets.zip`, upload as a private Kaggle Dataset (e.g. named `dissertation-eval-datasets`). Then in each analysis notebook, add that dataset as an input. The parquet files appear at `/kaggle/input/dissertation-eval-datasets/eval_*.parquet`.
* **Colab**: upload the zip, then in the analysis notebook: `!unzip eval_datasets.zip`. Files land in current working dir.

The `safe_load_first(...)` function inside `02` and `03` automatically looks for these local parquet files FIRST (checking several standard paths), and only falls back to fresh HuggingFace downloads if none are found.


In [ ]:
# =============================================================================
# BLOCK 0: PIP INSTALLS
# =============================================================================
!pip install -q -U "datasets>=2.14" pyarrow


In [ ]:
# =============================================================================
# BLOCK 1: DOWNLOAD + SAVE 7 EVAL DATASETS
# =============================================================================
import os, time, traceback
from datasets import load_dataset

SEED = 42
OUT_DIR = os.getcwd()
SAVED = []
FAILED = []


def _save_one(loader_callables, label):
    """Try each loader_callable in order. Save the first successful result as parquet."""
    last_err = None
    for i, fn in enumerate(loader_callables):
        try:
            print(f"  trying loader #{i} for {label} ...")
            ds = fn()
            path = os.path.join(OUT_DIR, f"eval_{label}.parquet")
            ds.to_parquet(path)
            mb = os.path.getsize(path) / 1e6
            print(f"  ✓ {label}: {len(ds)} samples -> {path} ({mb:.2f} MB)")
            SAVED.append({"label": label, "n": len(ds), "path": path, "loader_idx": i})
            return
        except Exception as e:
            last_err = e
            print(f"    loader #{i} failed: {type(e).__name__}: {str(e)[:200]}")
            continue
    print(f"  ✗ {label}: ALL LOADERS FAILED — {last_err}")
    FAILED.append({"label": label, "error": str(last_err)})


_t0 = time.perf_counter()

print("== truthfulqa ==")
_save_one([
    lambda: load_dataset("truthfulqa/truthful_qa", "generation", split="validation"),
    lambda: load_dataset("truthful_qa", "generation", split="validation"),
], "truthfulqa")

print("\n== triviaqa ==")
_save_one([
    lambda: load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation"),
    lambda: load_dataset("trivia_qa", "rc.nocontext", split="validation"),
], "triviaqa")

print("\n== coqa ==")
_save_one([
    lambda: load_dataset("stanfordnlp/coqa", split="validation"),
], "coqa")


print("\n== tydiqa ==")
def _load_tydi():
    ds = load_dataset("google-research-datasets/tydiqa", "secondary_task", split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())
def _load_tydi_legacy():
    ds = load_dataset("tydiqa", "secondary_task", split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())
_save_one([_load_tydi, _load_tydi_legacy], "tydiqa")


def _load_he(cfg):
    for cand in ("data", "train", "validation"):
        try: return load_dataset("pminervini/HaluEval", cfg, split=cand)
        except (ValueError, KeyError, FileNotFoundError): continue
    dd = load_dataset("pminervini/HaluEval", cfg)
    return dd[list(dd.keys())[0]]

for cfg, label in [("qa", "halueval_qa"), ("summarization", "halueval_summ"), ("dialogue", "halueval_dialog")]:
    print(f"\n== {label} ==")
    _save_one([lambda c=cfg: _load_he(c)], label)

# ---- 3 ADDITIONAL DATASETS (added 2026-05-30) -------------------------------
# Most-cited factual-QA benchmarks in 2024-25 hallucination papers.

print("\n== nq_open ==")
_save_one([
    lambda: load_dataset("google-research-datasets/nq_open", split="validation"),
    lambda: load_dataset("nq_open", split="validation"),
], "nq_open")

print("\n== hotpotqa ==")
_save_one([
    lambda: load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation", trust_remote_code=True),
    lambda: load_dataset("hotpot_qa", "distractor", split="validation", trust_remote_code=True),
], "hotpotqa")

print("\n== popqa ==")
_save_one([
    lambda: load_dataset("akariasai/PopQA", split="test"),
], "popqa")

print(f"\n=== DONE in {time.perf_counter() - _t0:.1f} s ===")
print(f"  saved: {len(SAVED)}/7")
print(f"  failed: {len(FAILED)}")
for f in FAILED:
    print(f"   ✗ {f['label']}: {f['error'][:150]}")


In [ ]:
# =============================================================================
# BLOCK 2: BUNDLE INTO ZIP (for one-click download / Kaggle Dataset creation)
# =============================================================================
import os, glob
files = sorted(glob.glob("eval_*.parquet"))
print(f"Found {len(files)} parquet files to bundle:")
for f in files:
    print(f"  {f}  ({os.path.getsize(f)/1e6:.2f} MB)")

# Use python's zipfile so it works on Kaggle too (no `!zip` dependency)
import zipfile
with zipfile.ZipFile("eval_datasets.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in files:
        zf.write(f)
print(f"\n✓ wrote eval_datasets.zip  ({os.path.getsize('eval_datasets.zip')/1e6:.2f} MB)")


In [ ]:
# =============================================================================
# BLOCK 3: PRINT NEXT STEPS
# =============================================================================
print("=" * 78)
print(" NEXT STEPS")
print("=" * 78)
print()
print(" 1. Download eval_datasets.zip from this session (Files panel, right-click > download)")
print()
print(" 2. KAGGLE workflow:")
print("    a) Datasets page > 'New Dataset' > drag eval_datasets.zip")
print("       Name it 'dissertation-eval-datasets' (private)")
print("    b) In each model's 02 and 03 notebook: Add Input > pick the dataset")
print("    c) Files will be at /kaggle/input/dissertation-eval-datasets/eval_*.parquet")
print("    d) Run the notebook normally; safe_load_first() picks them up automatically.")
print()
print(" 3. COLAB workflow:")
print("    a) In each model's 02 and 03 notebook, after BLOCK 0 (pip), add a cell:")
print("       !pip -q install gdown")
print("       # ...or just upload eval_datasets.zip manually to the file panel")
print("    b) Then run:  !unzip -o eval_datasets.zip")
print("    c) safe_load_first() picks them up from the working directory.")
print()
print(" 4. Verify after upload:")
print("    !ls -la eval_*.parquet")
print(" 5. If safe_load_first prints '(HF loader #0)' instead of '(LOCAL: ...)', upload missed.")
